Import Libraries

In [1]:
import pandas as pd
import numpy as np
import ast

Load BOTH datasets

In [2]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

Merge datasets

In [3]:
movies = movies.merge(credits, on='title')

Select useful columns

In [4]:
movies = movies[
    [
        'title',
        'genres',
        'cast',
        'crew',
        'release_date',
        'vote_average'
    ]
]

Remove null values

In [5]:
movies.dropna(inplace=True)

Extract year

In [6]:
movies['year'] = movies['release_date'].apply(
    lambda x: int(x.split("-")[0])
)

Convert genres

In [7]:
def convert(obj):

    L = []

    for i in ast.literal_eval(obj):
        L.append(i['name'])

    return L

movies['genres'] = movies['genres'].apply(convert)

Extract cast

In [8]:
def get_cast(obj):

    L = []
    counter = 0

    for i in ast.literal_eval(obj):

        if counter != 3:
            L.append(i['name'])
            counter += 1

        else:
            break

    return L

movies['cast'] = movies['cast'].apply(get_cast)

Extract director

In [9]:
def fetch_director(obj):

    L = []

    for i in ast.literal_eval(obj):

        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L

movies['crew'] = movies['crew'].apply(fetch_director)

Clean spaces

In [10]:
movies['genres'] = movies['genres'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

movies['cast'] = movies['cast'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

movies['crew'] = movies['crew'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

Create tags

In [11]:
movies['tags'] = (
    movies['genres'] +
    movies['cast'] +
    movies['crew']
)

Convert tags to text

In [12]:
movies['tags'] = movies['tags'].apply(
    lambda x:" ".join(x)
)

Vectorization

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors = cv.fit_transform(
    movies['tags']
).toarray()

Add year feature

In [14]:
year_feature = movies['year'].values.reshape(-1,1)

X = np.concatenate(
    (vectors, year_feature),
    axis=1
)

y = movies['vote_average']

Train-test split

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Import models

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_squared_error

Model comparison

In [17]:
models = {

    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(),

    "Random Forest": RandomForestRegressor(),

    "Gradient Boosting": GradientBoostingRegressor()
}

Train & compare ALL models

In [18]:
results = {}

for name, model in models.items():

    # TRAIN
    model.fit(X_train, y_train) 

    # PREDICT
    y_pred = model.predict(X_test)

    # MSE
    mse = mean_squared_error(y_test, y_pred)

    results[name] = mse

    print(f"{name} MSE: {mse}")

Linear Regression MSE: 8.309605081319155
Decision Tree MSE: 1.3785934973434972
Random Forest MSE: 1.0888252615106293
Gradient Boosting MSE: 1.0915638589704302


RandomForestRegressor()

In [19]:
from sklearn.ensemble import RandomForestRegressor

# FINAL MODEL
model = RandomForestRegressor()

# TRAIN
model.fit(X_train, y_train)

# PREDICT
y_pred = model.predict(X_test)

In [20]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

mse = mean_squared_error(y_test, y_pred)

mae = mean_absolute_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)

print("MSE :", mse)
print("MAE :", mae)
print("R2 Score :", r2)

MSE : 1.1008864861545806
MAE : 0.7032939428439428
R2 Score : 0.1288292163491045


In [21]:
import joblib

# =========================
# SAVE MODEL
# =========================
joblib.dump(model, "movie_model.pkl")

# =========================
# SAVE VECTORIZER
# =========================
joblib.dump(cv, "vectorizer.pkl")

print("✅ Model and Vectorizer Saved Successfully")

✅ Model and Vectorizer Saved Successfully


In [22]:
import os

print(os.listdir())

['.git', '.ipynb_checkpoints', 'app.py', 'movies.ipynb', 'movie_model.pkl', 'requirements.txt', 'tmdb_5000_credits.csv', 'tmdb_5000_movies.csv', 'vectorizer.pkl']
